In [1]:
import itertools
import typing as tp
from pathlib import Path

import pandas as pd
import yaml

In [2]:
outputs_dir = Path('../../outputs')

In [3]:
class Method:
    def __init__(self, config: tp.Dict[str, tp.Any], runs_dirs: tp.List[Path]) -> None:
        assert all(run_dir.is_dir() for run_dir in runs_dirs)        
        self._config = config
        self._metrics_dirs = list(itertools.chain(*(run_dir.glob('*/agg_metrics') for run_dir in runs_dirs)))
        
    @property
    def name(self) -> str:
        model_path = self._config['model_vec']['_target_']
        return model_path.rsplit('.')[-1]
    
    def get_results(self, metric_name: str) -> pd.DataFrame:
        metric_dfs = []
        
        for metrics_dir in self._metrics_dirs:
            metric_path = (metrics_dir / metric_name).with_suffix('.tsv')
            if metric_path.exists():
                metric_df = pd.read_csv(metric_path, sep='\t')
                metric_dfs.append(metric_df)
                
        if not metric_dfs:
            raise ValueError(f"No results for metric '{metric_name}'")
            
        metric_df = pd.concat(metric_dfs, ignore_index=True)
        by_part_df = metric_df.groupby('part')
    
        part_score_series = by_part_df['score'].nunique()
        assert all(part_score_series == 1)
        
        unique_metric_df = by_part_df.first().reset_index()
        return unique_metric_df
    
    def get_result(self, metric_name: str, part_name: str) -> float:
        metric_df = self.get_results(metric_name)
        part_metric_df = metric_df[metric_df['part'] == part_name]
        assert len(part_metric_df) == 1, f"No or duplicate results for the part='{part_name}', metric={metric_name}"
        return part_metric_df.iloc[0]['score']  # type: ignore
    
    def get_evaluated_metrics(self) -> tp.Set[str]:
        metrics_names = set()
        for metrics_dir in self._metrics_dirs:
            for metric_path in metrics_dir.iterdir():
                metric_name = str(metric_path.with_suffix('').name)
                metrics_names.add(metric_name)
        return metrics_names

In [4]:
def load_config(method_dir: Path) -> tp.Dict[str, tp.Any]:
    config_path = method_dir / '.hydra' / 'config.yaml'
    assert config_path.exists()
    
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
        
    config['eval_params'].pop('list_paths_datasets')
    return config

In [5]:
def load_methods(outputs_dir: Path) -> tp.Generator[Method, None, None]:
    assert outputs_dir.is_dir()
    
    methods_configs, methods_dirs = [], []
    runs_dirs = outputs_dir.glob('*/*/agg_metrics')
    
    for method_dir in runs_dirs:
        run_dir = method_dir.parent.parent
        method_config = load_config(run_dir)
        if method_config in methods_configs:
            method_ind = methods_configs.index(method_config)
            methods_dirs[method_ind].append(run_dir)
        else:
            methods_configs.append(method_config)
            methods_dirs.append([run_dir])
    
    for method_config, method_dirs in zip(methods_configs, methods_dirs):
        yield Method(method_config, method_dirs)

In [6]:
name_to_method = {method.name: method for method in load_methods(outputs_dir)}

## Results per metric

In [7]:
def metric_results(name_to_method: tp.Dict[str, Method], metric_name: str) -> pd.DataFrame:
    records = []
    
    for method in name_to_method.values():
        method_records = method.get_results(metric_name)[['score', 'part']].to_dict('records')
        method_part_to_score = {record['part']: record['score'] for record in method_records}
        records.append(method_part_to_score)
        
    metric_df = pd.DataFrame.from_records(records)
    metric_df.set_index(name_to_method.keys())
    metric_df = metric_df.T
    return metric_df

In [8]:
def pretty_df(df: pd.DataFrame, caption: tp.Optional[str] = None) -> tp.Any:
    styles = [
        dict(selector="th", props=[("font-size", "110%"), ("text-align", "center")]),
        dict(selector="caption", props=[("font-size", "150%")])
    ]
    properties = {'font-size': '120%'}
    return df.style.set_caption(caption).set_table_styles(styles).set_properties(**properties)

In [9]:
def show_metric_results(metric_name: str) -> tp.Any:
    results_df = metric_results(name_to_method, metric_name)
    return pretty_df(results_df, metric_name)

## Results per dataset

In [10]:
def dataset_results(name_to_method: tp.Dict[str, Method], considered_metrics: tp.List[str], dataset: str) \
        -> pd.DataFrame:
    records = []
    
    for method in name_to_method.values():
        method_records = {}
        for metric in considered_metrics:
            metric_df = method.get_results(metric)
            dataset_df = metric_df[metric_df['part'] == dataset]
            if len(dataset_df) > 0:
                assert len(dataset_df) == 1
                method_records[metric] = dataset_df.iloc[0]['score']
        records.append(method_records)
        
    metric_df = pd.DataFrame.from_records(records)
    metric_df.set_index(name_to_method.keys())
    return metric_df

In [11]:
ari_metrics = [
    'fh_max_ARI', 'max_ARI', 'calinski_harabasz_ARI', 'silhouette_ARI'
]
s10_metrics = [
    'max_S10_AVG', 'fh_max_S10_AVG', 'max_S10_Completeness', 'max_S10_Homogeneity',
    'silhouette_S10_AVG', 'calinski_harabasz_S10_AVG'
]

In [12]:
names_cuts = {
    'calinski_harabasz': 'ch',
    'silhouette': 'sil',
    'Completeness': 'Compl',
    'Homogeneity': 'Homo'
}

def cut_name(name: str) -> str:
    for from_str, to_str in names_cuts.items():
        name = name.replace(from_str, to_str)
    return name

In [13]:
def show_dataset_results(dataset: str, metrics: tp.List[str]) -> tp.Any:
    results_df = dataset_results(name_to_method, metrics, dataset)
    results_df.columns = [cut_name(column_name) for column_name in results_df.columns]
    return pretty_df(results_df, dataset)

## internal table representation

In [32]:
run_dir = outputs_dir / "2022-08-26_16-12-53"

In [33]:
method_config = load_config(run_dir)
# Additional runs dirs can be added. The results will be merged
method = Method(method_config, [run_dir])
method.name

'GLMVectorizer'

In [34]:
# From Python 3.7 dict preserves the order
table_columns_structure = {
    'se10': {'en': ['semeval10']},
    'se13': {'en': ['semeval13']},
    'russe_bts-rnc': {'ru': ['train', 'test-public', 'test-private']},
    'xl_wsd': {'en': ['dev'], 'multilang': ['dev', 'test'], 'zh': ['dev', 'test']},
}

In [35]:
def compute_method_metric_table(global_metric_name: str, metrics_prefixes: tp.List[str],
                                prefixes_shortcuts: tp.Optional[tp.Dict[str, str]] = None,
                                rounding: tp.Optional[int] = 1) -> pd.DataFrame:
    index_values, metric_values = [], []

    for dataset_name, langs_dict in table_columns_structure.items():
        for lang, parts_list in langs_dict.items():
            for part_name in parts_list:
                part_id = f'{dataset_name}-{lang}-{part_name}'
                for metric_prefix in metrics_prefixes:
                    metric_name = f'{metric_prefix}{global_metric_name}'
                    metric_shortcut = metric_prefix
                    if prefixes_shortcuts and metric_prefix in prefixes_shortcuts:
                        metric_shortcut = prefixes_shortcuts[metric_prefix]
                    index_values.append((dataset_name, lang, part_name, metric_shortcut))
                    metric_value = method.get_result(metric_name, part_id)
                    if rounding is not None:
                        metric_value = round(metric_value, rounding)
                    metric_values.append(metric_value)

    multi_index = pd.MultiIndex.from_tuples(index_values)
    return pd.DataFrame([metric_values], columns=multi_index, index=[global_metric_name])

In [36]:
methods_metrics_prefixes = ['calinski_harabasz_', 'silhouette_']
bounds_metrics_prefixes = ['max_', 'fh_max_']

prefixes_shortcuts = {'calinski_harabasz_': 'cal_har_', 'silhouette_': 'sil_'}

In [37]:
global_metrics = [
    'ARI', 
    'S10_FScore', 'S10_VMeasure', 'S10_AVG',
    'S13_F1', 'S13_FNMI', 'S13_AVG'
]

In [38]:
def compute_method_table(global_metrics: tp.Iterable[str], metrics_prefixes: tp.List[str],
                         prefixes_shortcuts: tp.Optional[tp.Dict[str, str]] = None,
                         rounding: tp.Optional[int] = 1) -> pd.DataFrame:
    metrics_dfs = [
        compute_method_metric_table(metric, metrics_prefixes, prefixes_shortcuts, rounding)
        for metric in global_metrics
    ]
    return pd.concat(metrics_dfs)

In [39]:
compute_method_table(global_metrics, methods_metrics_prefixes, prefixes_shortcuts)

se10            se13       russe_bts-rnc                    \
                    en              en                  ru                     
             semeval10       semeval13               train       test-public   
              cal_har_  sil_  cal_har_  sil_      cal_har_  sil_    cal_har_   
ARI               48.2  39.8      37.5  28.0          57.6  47.3        58.9   
S10_FScore        73.3  73.3      65.7  65.2          82.5  82.9        82.1   
S10_VMeasure      48.8  41.4      42.2  31.0          55.4  46.8        53.4   
S10_AVG           59.8  55.1      52.7  45.0          67.6  62.3        66.2   
S13_F1            73.0  75.4      69.3  69.7          83.1  82.7        81.7   
S13_FNMI          39.2  37.1      23.6  17.2          45.6  35.0        44.9   
S13_AVG           53.5  52.9      40.4  34.6          61.6  53.8        60.6   

                                        xl_wsd                                 \
                                            en       multilang                  
                   test-private            dev             dev           test   
              sil_     cal_har_  sil_ cal_har_  sil_  cal_har_  sil_ cal_har_   
ARI           54.9         54.3  50.9     38.3  26.5      24.3  17.6     20.7   
S10_FScore    84.6         79.3  82.5     63.9  67.9      49.5  53.2     47.3   
S10_VMeasure  49.7         53.4  49.7     50.7  38.0      42.9  32.0     38.0   
S10_AVG       64.9         65.1  64.0     56.9  50.8      46.1  41.2     42.4   
S13_F1        85.1         81.1  83.5     64.5  65.8      44.5  50.1     44.8   
S13_FNMI      41.6         46.0  42.6     36.7  26.6      30.7  20.9     26.1   
S13_AVG       59.5         61.1  59.6     48.6  41.8      37.0  32.3     34.2   

                                                  
                         zh                       
                        dev           test        
              sil_ cal_har_  sil_ cal_har_  sil_  
ARI           18.3     34.4  32.6     31.4  24.5  
S10_FScore    53.1     47.2  51.2     46.0  44.5  
S10_VMeasure  29.6     60.5  54.9     57.3  46.5  
S10_AVG       39.6     53.4  53.0     51.3  45.5  
S13_F1        51.4     54.7  59.4     54.1  53.4  
S13_FNMI      21.8     39.8  38.8     37.0  31.5  
S13_AVG       33.5     46.6  48.0     44.8  41.0

In [40]:
compute_method_table(global_metrics, bounds_metrics_prefixes)

se10              se13         russe_bts-rnc          \
                    en                en                    ru           
             semeval10         semeval13                 train           
                  max_ fh_max_      max_ fh_max_          max_ fh_max_   
ARI               62.2    45.1      53.1    42.0          75.7    59.0   
S10_FScore        81.3    73.0      73.4    67.6          90.3    84.1   
S10_VMeasure      60.9    49.0      55.9    49.6          69.1    56.5   
S10_AVG           69.8    59.7      63.4    57.0          78.5    68.4   
S13_F1            81.7    74.4      74.4    69.8          88.3    83.3   
S13_FNMI          55.2    39.3      34.5    27.7          61.5    44.5   
S13_AVG           66.4    53.7      49.3    42.9          73.2    60.8   

                                                      xl_wsd          \
                                                          en           
             test-public         test-private            dev           
                    max_ fh_max_         max_ fh_max_   max_ fh_max_   
ARI                 77.8    65.0         73.9    59.9   72.1    48.8   
S10_FScore          91.1    85.5         89.3    84.5   85.5    74.0   
S10_VMeasure        69.6    57.2         69.8    57.6   77.0    61.2   
S10_AVG             79.3    69.9         78.3    69.6   81.0    66.3   
S13_F1              89.6    84.6         88.2    84.9   77.1    68.6   
S13_FNMI            61.4    46.4         61.3    45.9   62.7    41.7   
S13_AVG             73.9    62.0         73.1    62.1   68.6    53.0   

                                                                          
             multilang                          zh                        
                   dev          test           dev          test          
                  max_ fh_max_  max_ fh_max_  max_ fh_max_  max_ fh_max_  
ARI               49.4    24.6  46.3    26.5  66.4    45.4  60.8    39.9  
S10_FScore        69.1    52.8  66.6    54.9  71.7    54.8  66.0    49.0  
S10_VMeasure      65.4    50.2  63.3    50.6  83.9    73.2  81.6    74.1  
S10_AVG           66.2    48.5  63.6    48.7  77.0    63.0  72.6    58.4  
S13_F1            58.6    49.4  58.2    52.3  71.7    59.8  68.5    55.4  
S13_FNMI          50.9    33.9  52.0    34.0  69.2    51.6  63.6    48.0  
S13_AVG           53.2    37.6  52.5    40.8  69.0    54.1  64.6    50.4